In [52]:
import os
import shutil
from pathlib import Path

from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, LongType, DateType
from pyspark.sql import Row
from pyspark.sql.functions import expr
from pyspark.sql.functions import *

spark = (
    SparkSession.builder
    .appName("Spark SQL Practice")
    .config("spark.master", "local[*]")
    .config("spark.sql.warehouse.dir", "src/main/resources/warehouse") # this will specify where to store our tables
    .config("spark.sql.legacy.allowCreatingMangedTableUsingNonemptyLocation", "true")
    .getOrCreate()
)


26/03/22 03:06:47 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


In [53]:
carsDF = spark.read.json("src/main/resources/data/cars.json")

carsDF.show(5)

+------------+---------+------------+----------+----------------+--------------------+------+-------------+----------+
|Acceleration|Cylinders|Displacement|Horsepower|Miles_per_Gallon|                Name|Origin|Weight_in_lbs|      Year|
+------------+---------+------------+----------+----------------+--------------------+------+-------------+----------+
|        12.0|        8|       307.0|       130|            18.0|chevrolet chevell...|   USA|         3504|1970-01-01|
|        11.5|        8|       350.0|       165|            15.0|   buick skylark 320|   USA|         3693|1970-01-01|
|        11.0|        8|       318.0|       150|            18.0|  plymouth satellite|   USA|         3436|1970-01-01|
|        12.0|        8|       304.0|       150|            16.0|       amc rebel sst|   USA|         3433|1970-01-01|
|        10.5|        8|       302.0|       140|            17.0|         ford torino|   USA|         3449|1970-01-01|
+------------+---------+------------+----------+

In [54]:
# spark sql

# this creates an alias in spark so i can refer to this dataframe as a table
carsDF.createOrReplaceTempView("cars")


# whenever u run sql u return a dataframe
americanCarsDF = spark.sql(
    """
        select Name from cars
        where Origin = 'USA'
    """
)

americanCarsDF.show(5)

+--------------------+
|                Name|
+--------------------+
|chevrolet chevell...|
|   buick skylark 320|
|  plymouth satellite|
|       amc rebel sst|
|         ford torino|
+--------------------+
only showing top 5 rows



In [55]:
# this returns an empty dataframe, and creates a new folder in our project 
# src\main\resources\warehouse\rtjvm.db 
# this folder will store all our databases!!
spark.sql("DROP DATABASE IF EXISTS rtjvm CASCADE")
spark.sql("CREATE DATABASE rtjvm")

DataFrame[]

In [56]:
# now every subsequent select will be related to the rtjvm database
spark.sql("use rtjvm")

DataFrame[]

In [57]:
# see all databases
databasesDF = spark.sql("show databases")

databasesDF.show()

+---------+
|namespace|
+---------+
|  default|
|    rtjvm|
+---------+



In [58]:
# transfer tables from a database to a spark table
employeesDB = (
    spark.read
    .format("jdbc")
    .option("driver", "org.postgresql.Driver")
    .option("user", "docker")
    .option("password", "docker")
    .option("url", "jdbc:postgresql://postgres:5432/rtjvm")
    .option("dbtable", "public.employees")
    .load()
)

# this will SAVE employees as a table into the current database we are using from the `use rtjvm` command
# after this runs we should see an employees folder within our warehouse
# src\main\resources\warehouse\rtjvm.db
(
    employeesDB.write
    .mode("Overwrite")
    .saveAsTable("employees")
)

In [59]:
def readTable(tableName):
   return (
    spark.read
    .format("jdbc")
    .option("driver", "org.postgresql.Driver")
    .option("user", "docker")
    .option("password", "docker")
    .option("url", "jdbc:postgresql://postgres:5432/rtjvm")
    .option("dbtable", "public.employees")
    .load())


def transferTables(tableNames):
   for tableName in tableNames:
      tableDF = readTable(tableName)
      # load it into memory so we can refer to it in spark sql
      tableDF.createOrReplaceGlobalTempView(tableName)
      # save as table
      tableDF.write.mode("Overwrite").saveAsTable(tableName)
 

In [ ]:
# transfer all tables to our warehouse - so from regular database to spark table (datawarehouse)
transferTables(["employees", "departments", "titles", "dept_emp", "salaries", "dept_manager"])

In [ ]:
# read df from warehouse
employeesDF2 = spark.read.table("employees")

# now you can do whatever you want with this dataframe sql or regular dataframe transformations